In [ ]:
# ============================================================
# KD-TIES: Cross-Manifold LoRA Merging in Knowledge Distillation
# Diagnosing TIES Failure Modes and Evaluating Task Vector Corrections
#
# Author: Rishav Raj | Roll: 2024MSCS014
# Central University of Rajasthan | June 2026
#
# EXECUTION NOTE:
# This notebook requires two-phase execution.
# Phase 1-2: Data generation and LoRA training (this session)
# Phase 3-4: Merge, evaluation, subspace analysis (restart kernel first)
# vLLM and Unsloth cannot share a Python session.
# Minimum hardware: A100 80GB VRAM, 30GB CPU RAM, 50GB disk
# ============================================================

import os
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"Working directory: {os.getcwd()}")

In [ ]:
# Phase 1 uses vLLM only
# Do NOT install Unsloth in this session

import subprocess
subprocess.run([
    "pip", "install", "vllm==0.4.2",
    "--quiet"
])

print("vLLM installed. Do not import Unsloth in this session.")

# Phase 1 — Distillation Data Generation
#
# Teacher:  Qwen2.5-32B-Instruct
# Domain 1: GPT-4 instruction dataset (242K total, 5000 sampled) → D1
# Domain 2: MetaMathQA GSM_AnsAug subset (395K total, 5000 sampled) → D2
#
# Method: vLLM batch inference, greedy decoding (temperature=0.0)
# Output: Two JSONL files in Qwen2.5 chat template format
# Time:   ~15 minutes per domain on A100****

In [ ]:
from vllm import LLM, SamplingParams

TEACHER_MODEL = "Qwen/Qwen2.5-32B-Instruct"

print(f"Loading teacher: {TEACHER_MODEL}")
print("This requires ~65GB VRAM. Estimated load time: 3-5 minutes.")

llm = LLM(
    model=TEACHER_MODEL,
    tensor_parallel_size=1,
    gpu_memory_utilization=0.92,
    dtype="bfloat16",
    trust_remote_code=True,
)

sampling_params = SamplingParams(
    temperature=0.0,     # greedy decoding — deterministic
    max_tokens=1024,
    top_p=1.0,
)

print("Teacher loaded successfully.")

In [ ]:
from datasets import load_dataset
import random

random.seed(42)

print("Loading GPT-4 instruction dataset...")
d1_raw = load_dataset(
    "teknium/GPT4-LLM-Cleaned",
    split="train"
)

print(f"Total entries: {len(d1_raw)}")

# Sample 5000
d1_indices = random.sample(range(len(d1_raw)), 5000)
d1_sample  = d1_raw.select(d1_indices)

d1_instructions = [item["instruction"] for item in d1_sample]

print(f"Sampled: {len(d1_instructions)} instructions")
print(f"Example: {d1_instructions[0][:100]}")

In [ ]:
import json

CHAT_TEMPLATE = "<|im_start|>user\n{instruction}<|im_end|>\n<|im_start|>assistant\n"

# Format prompts
d1_prompts = [
    CHAT_TEMPLATE.format(instruction=instr)
    for instr in d1_instructions
]

print(f"Generating D1 responses from teacher...")
print(f"Batch size: {len(d1_prompts)} prompts")

d1_outputs = llm.generate(d1_prompts, sampling_params)

# Save to disk
d1_path = "/kaggle/working/d1_instruction.jsonl"

with open(d1_path, "w") as f:
    for i, output in enumerate(d1_outputs):
        entry = {
            "instruction": d1_instructions[i],
            "response":    output.outputs[0].text,
        }
        f.write(json.dumps(entry) + "\n")

print(f"D1 saved: {d1_path}")
print(f"File size: {os.path.getsize(d1_path) / 1e6:.1f} MB")

In [ ]:
print("Loading MetaMathQA dataset...")
d2_raw = load_dataset(
    "meta-math/MetaMathQA",
    split="train"
)

# Filter GSM_AnsAug subset only
d2_gsm = d2_raw.filter(
    lambda x: x["type"] == "GSM_AnsAug"
)

print(f"Total GSM_AnsAug entries: {len(d2_gsm)}")

# Sample 5000
d2_indices = random.sample(range(len(d2_gsm)), 5000)
d2_sample  = d2_gsm.select(d2_indices)

d2_questions = [item["query"] for item in d2_sample]

print(f"Sampled: {len(d2_questions)} questions")
print(f"Example: {d2_questions[0][:100]}")

In [ ]:
d2_prompts = [
    CHAT_TEMPLATE.format(instruction=q)
    for q in d2_questions
]

print(f"Generating D2 responses from teacher...")

d2_outputs = llm.generate(d2_prompts, sampling_params)

d2_path = "/kaggle/working/d2_math.jsonl"

with open(d2_path, "w") as f:
    for i, output in enumerate(d2_outputs):
        entry = {
            "instruction": d2_questions[i],
            "response":    output.outputs[0].text,
        }
        f.write(json.dumps(entry) + "\n")

print(f"D2 saved: {d2_path}")
print(f"File size: {os.path.getsize(d2_path) / 1e6:.1f} MB")

In [ ]:
def verify_dataset(path, name):
    with open(path, "r") as f:
        lines = f.readlines()
    entries = [json.loads(l) for l in lines]
    print(f"\n{name}")
    print(f"  Entries:  {len(entries)}")
    print(f"  Example instruction: {entries[0]['instruction'][:80]}")
    print(f"  Example response:    {entries[0]['response'][:80]}")

verify_dataset(d1_path, "D1 — Instruction Domain")
verify_dataset(d2_path, "D2 — Math Domain")

print("\nPhase 1 complete. Both datasets verified.")
print("Next: restart kernel, then run Phase 2 (LoRA training).")

In [ ]:
import gc

del llm
gc.collect()
torch.cuda.empty_cache()

print(f"VRAM freed.")
print(f"VRAM available: {torch.cuda.memory_reserved(0) / 1e9:.1f} GB")
print("\nRESTART KERNEL before running Phase 2.")
print("Do not import Unsloth in this session.")